# Feature Detection: Hough Transform (Line Detection)

This notebook documents the Hough transform for detecting collinear structures in digital images. The method is robust to noise and edge fragmentation, making it a standard tool in vision systems for robotics and autonomous navigation.

Topics covered:
- Spatial duality: mapping points from the Cartesian domain $(x, y)$ to sinusoids in polar parameter space $(\rho, \theta)$.
- Peak detection in the accumulator space.
- Standard Hough Transform (SHT) for projecting analytical, infinite lines.
- Performance optimization with the Probabilistic Hough Transform (PHT) for extracting finite line segments.
- Preprocessing pipelines for real-world scenes (structural noise reduction).

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from skimage.transform import hough_line, hough_line_peaks
import os
import glob

# Download required datasets
if not os.path.exists("kwadraty.png") :
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/11_Hough/kwadraty.png --no-check-certificate
if not os.path.exists("lab112.png") :
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/11_Hough/lab112.png --no-check-certificate
if not os.path.exists("dom.png") :
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/11_Hough/dom.png --no-check-certificate

# Initialize a matrix with a single point (Dirac impulse)
im = np.zeros((64,64), dtype=np.uint8)
im[18, 31] = 1

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(im, 'gray')
ax.set_title("Single point in image space")
ax.axis('off')
plt.show()


## 1. Mapping to Accumulator Space

A single point in the image domain maps to a sinusoid in Hough space (infinitely many lines pass through one point). Finding a line through several points reduces to detecting the intersection of their sinusoids in $(\rho, \theta)$ space.

In [ ]:
def show_hough(h, image, title_img='Image domain (x, y)', title_h='Hough space (\u03c1, \u03b8)'):
    """Visualization tool for analyzing transform duality."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    ax = axes.ravel()

    ax[0].imshow(image, 'gray')
    ax[0].set_title(title_img)
    ax[0].set_axis_off()

    ax[1].imshow(h, 'gray', aspect='auto')
    ax[1].set_title(title_h)
    ax[1].set_xlabel('Angle \u03b8 (degrees)')
    ax[1].set_ylabel('Distance \u03c1 (pixels)')
    
    plt.tight_layout()
    plt.show()    

h, theta, d = hough_line(im)
show_hough(h, im)

### Superposition and Collinearity
Experiment confirming accumulation: four collinear points in Cartesian space produce four sinusoids intersecting at a single peak in parameter space.

In [ ]:
im2 = np.zeros((64,64), dtype=np.uint8)
im2[18, 31] = 1
im2[40, 50] = 1
h2, _, _ = hough_line(im2)
show_hough(h2, im2, title_img="Two points (one connecting line)")

im3 = np.zeros((64,64), dtype=np.uint8)
im3[10, 10] = 1
im3[20, 20] = 1
im3[30, 30] = 1
im3[40, 40] = 1
h3, _, _ = hough_line(im3)
show_hough(h3, im3, title_img="Collinear set (clear global maximum)")

## 2. Linear Structure Detection (SHT)

In production systems, the Hough transform is applied to an edge image (e.g., Canny output). This restricts the input to pixels with strong structural evidence.

In [ ]:
kw = cv2.imread('kwadraty.png', 0)
edges = cv2.Canny(kw, 50, 150)
h_kw, theta_kw, d_kw = hough_line(edges)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(edges, cmap='gray')
ax[0].set_title("Input: Edges (Canny)")
ax[0].axis('off')
ax[1].imshow(h_kw, cmap='gray', aspect='auto')
ax[1].set_title("Accumulator matrix (8 visible peaks)")
plt.show()

# Automatic peak detection
peaks = hough_line_peaks(h_kw, theta_kw, d_kw)

fig, ax = plt.subplots(1, figsize=(8, 8))
ax.set_aspect('auto')
ax.imshow(h_kw, 'gray', aspect='auto')
ax.set_title("Peak locations")

for _, angle, dist in zip(*peaks):
    x = np.argmin(np.abs(theta_kw - angle))
    y = np.argmin(np.abs(d_kw - dist))
    circle = plt.Circle((x, y), 10, color='r', fill=False, linewidth=2)
    ax.add_patch(circle)

plt.show()

### Inverse Transform
Project analytical (infinite) lines found in the Hough matrix back onto the base RGB image.

In [ ]:
kw_c = cv2.imread('kwadraty.png')
lines = cv2.HoughLines(edges, 1, np.pi/180, 60)

for line in lines:
    rho, theta = line[0]
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = a*rho
    y0 = b*rho
    # Project ±1000 px from the base point
    x1 = int(x0 + 1000*(-b))
    y1 = int(y0 + 1000*(a))
    x2 = int(x0 - 1000*(-b))
    y2 = int(y0 - 1000*(a))
    cv2.line(kw_c, (x1, y1), (x2, y2), (0, 0, 255), 2)

plt.figure(figsize=(8,8))
plt.imshow(kw_c[:,:,::-1])
plt.title("Standard Hough Transform (SHT) - Infinite lines")
plt.axis('off')
plt.show()

## 3. Probabilistic Hough Transform (PHT)

An extension of the base algorithm that stochastically samples edge space. It is computationally efficient and, via internal parameters (`minLineLength`, `maxLineGap`), estimates segment endpoints by grouping broken edge chains into single logical vectors.

In [ ]:
kw_c2 = cv2.imread('kwadraty.png')
linesP = cv2.HoughLinesP(edges, 1, np.pi/180, 40, minLineLength=20, maxLineGap=10)

for line in linesP:
    x1, y1, x2, y2 = line[0]
    cv2.line(kw_c2, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(8,8))
plt.imshow(kw_c2[:,:,::-1])
plt.title("Probabilistic Hough Transform (PHT) - Finite segments")
plt.axis('off')
plt.show()

## 4. Real-World (Noisy) Scene Pipeline

Processing natural scenes typically requires a preprocessing pipeline. Structural noise and fine texture artifacts (e.g., roof tiles) degrade Hough space and cause false positives.

In [ ]:
# Controlled-noise scene analysis (maze structure)
lab = cv2.imread('lab112.png', 0)
lab_c = cv2.imread('lab112.png')

_, thresh = cv2.threshold(lab, 40, 255, cv2.THRESH_BINARY)
edges_lab = cv2.Canny(thresh, 50, 150)

h_l, theta_l, d_l = hough_line(edges_lab)
peaks_l = hough_line_peaks(h_l, theta_l, d_l)

for _, angle, dist in zip(*peaks_l):
    a = np.cos(angle)
    b = np.sin(angle)
    x0 = a*dist
    y0 = b*dist
    x1 = int(x0 + 1000*(-b))
    y1 = int(y0 + 1000*(a))
    x2 = int(x0 - 1000*(-b))
    y2 = int(y0 - 1000*(a))
    cv2.line(lab_c, (x1, y1), (x2, y2), (0, 255, 0), 2)

plt.figure(figsize=(10,6))
plt.imshow(lab_c[:,:,::-1])
plt.title("Perspective detection - Laboratory environment (SHT)")
plt.axis('off')
plt.show()

print("="*80)

# Complex scene analysis (Gaussian blur with LPF preprocessing)
dom = cv2.imread('dom.png', 0)
dom_c = cv2.imread('dom.png')

blur = cv2.GaussianBlur(dom, (5, 5), 0)
edges_dom = cv2.Canny(blur, 50, 150)

lines_dom = cv2.HoughLinesP(edges_dom, 1, np.pi/180, 30, minLineLength=20, maxLineGap=5)

for line in lines_dom:
    x1, y1, x2, y2 = line[0]
    cv2.line(dom_c, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(10,6))
plt.imshow(dom_c[:,:,::-1])
plt.title("Architectural outline extraction - LPF preprocessing + PHT")
plt.axis('off')
plt.show()

### Workspace Cleanup

In [ ]:
trash_files = glob.glob('kwadraty.png') + glob.glob('lab112.png') + glob.glob('dom.png')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Working directory cleaned of temporary files.")